# 03 — Clustering & Validation
HCA to classify seasons → RF to validate.

**Input**: `data/feature_matrix.csv`  
**Output**: `data/feature_matrix_clustered.csv`, Figures 3 & 4

In [ ]:
import pandas as pd
import numpy as np
import os, sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

from src.clustering import run_hca, plot_optimization, plot_pca, identify_cluster_types
from src.validation import validate_rf
print('Imports ready')

In [ ]:
# ── Load feature matrix from notebook 02 ──
fm = pd.read_csv('data/feature_matrix.csv')
print(f'Feature matrix: {fm.shape}')
os.makedirs('figures', exist_ok=True)

In [ ]:
# ── Step 1: Run HCA with k=3 ──
labels, Z, X_scaled, feat_cols = run_hca(fm, n_clusters=3)
fm['cluster'] = labels

In [ ]:
# ── Step 2: Optimization plots (elbow + silhouette) ──
plot_optimization(X_scaled)

In [ ]:
# ── Step 3: PCA visualization (Figure 3A) ──
plot_pca(X_scaled, labels)

In [ ]:
# ── Step 4: Identify which cluster is Cool / PRE-V / POST-V ──
identify_cluster_types(fm)

In [ ]:
# ── Step 5: Set the mapping ──
# ⚠️ ADJUST these numbers based on identify_cluster_types() output above!
# Look at: cluster size, MayJul vs AugOct heat_days, and total heat_days

cluster_map = {
    1: 'POST-V',   # smallest, highest AugOct heat → adjust if needed
    2: 'PRE-V',    # medium, highest MayJul heat   → adjust if needed
    3: 'Cool'      # largest, ~0 heat days          → adjust if needed
}

fm['season_type'] = fm['cluster'].map(cluster_map)
print(fm['season_type'].value_counts())
print(f'\nPaper expects: Cool ~124, PRE-V ~70, POST-V ~21')

In [ ]:
# ── Step 6: RF validation (Figure 4) — takes ~2 min ──
ters, imp_df = validate_rf(X_scaled, labels, feat_cols)

In [ ]:
# ── Save clustered feature matrix ──
fm.to_csv('data/feature_matrix_clustered.csv', index=False)
print(f'Saved: data/feature_matrix_clustered.csv ({fm.shape})')

### Checkpoint
At this point you should have:
- ✅ 3 clusters with sizes close to 21 / 70 / 124
- ✅ PCA plot showing clear separation
- ✅ RF TER close to 1.35%
- ✅ Top features are heat-related (heatwaves, heat days, VPD)

**Next → Notebook 04: Yield & Quality Analysis**